# Lab 4: Initial Table Setup

This notebook contains one-time initialization steps.
Run the required cells manually before running the pipeline job.

Do not include this notebook in the scheduled job.
The SCD2 initialization cell must not be rerun after the table has been created.

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("silver_schema", "viktoriia_kalenichenko_silver")
dbutils.widgets.text("scd2_table", "dim_country_scd2_v2")

catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")

scd2_table = (
    f"{catalog}.{silver_schema}."
    f"{dbutils.widgets.get('scd2_table')}"
)

silver_table = f"{catalog}.{silver_schema}.environment"

## Prepare the Initial Country Dimension

Read country identifiers from Silver and enrich them with the
reference region and subregion mapping.

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

country_df = (
    spark.table(f"{catalog}.{silver_schema}.environment")
    .select("AREA","country_code")
    .dropDuplicates(["AREA"]))

#reference mapping (list)
country_regions = [
    ("Albania", "Europe", "Southern Europe"),
    ("Austria", "Europe", "Western Europe"),
    ("Belarus", "Europe", "Eastern Europe"),
    ("Belgium", "Europe", "Western Europe"),
    ("Bosnia and Herzegovina", "Europe", "Southern Europe"),
    ("Bulgaria", "Europe", "Eastern Europe"),
    ("Croatia", "Europe", "Southern Europe"),
    ("Czechia", "Europe", "Central Europe"),
    ("Denmark", "Europe", "Northern Europe"),
    ("Estonia", "Europe", "Northern Europe"),
    ("Finland", "Europe", "Northern Europe"),
    ("France", "Europe", "Western Europe"),
    ("Germany", "Europe", "Western Europe"),
    ("Greece", "Europe", "Southern Europe"),
    ("Hungary", "Europe", "Central Europe"),
    ("Iceland", "Europe", "Northern Europe"),
    ("Ireland", "Europe", "Northern Europe"),
    ("Italy", "Europe", "Southern Europe"),
    ("Latvia", "Europe", "Northern Europe"),
    ("Lithuania", "Europe", "Northern Europe"),
    ("Luxembourg", "Europe", "Western Europe"),
    ("Malta", "Europe", "Southern Europe"),
    ("Moldova", "Europe", "Eastern Europe"),
    ("Montenegro", "Europe", "Southern Europe"),
    ("Netherlands", "Europe", "Western Europe"),
    ("North Macedonia", "Europe", "Southern Europe"),
    ("Norway", "Europe", "Northern Europe"),
    ("Poland", "Europe", "Central Europe"),
    ("Portugal", "Europe", "Southern Europe"),
    ("Romania", "Europe", "Eastern Europe"),
    ("Serbia", "Europe", "Southern Europe"),
    ("Slovak Republic", "Europe", "Central Europe"),
    ("Slovenia", "Europe", "Central Europe"),
    ("Spain", "Europe", "Southern Europe"),
    ("Sweden", "Europe", "Northern Europe"),
    ("Switzerland", "Europe", "Western Europe"),
    ("Ukraine", "Europe", "Eastern Europe"),
    ("United Kingdom", "Europe", "Northern Europe"),
]

regions_df = spark.createDataFrame(country_regions,["AREA", "region", "subregion"])

country_df = country_df.join(regions_df, on="AREA", how="left")

## Initialize the SCD Type 2 Table

Run this cell only when initializing a new SCD2 table.

In [0]:
initial_scd2_df = (
    country_df
    .withColumn("valid_from", F.current_timestamp())
    .withColumn("valid_to", F.lit(None).cast("timestamp"))
    .withColumn("is_current", F.lit(True))
)

(
    initial_scd2_df.write
    .format("delta")
    .mode("error")
    .saveAsTable(scd2_table)
)

## Configure the silver table layout

Configure Liquid Clustering on AREA and YEAR as an initial table
layout setting. Regular OPTIMIZE and VACUUM operations are performed
in the maintenance notebook.

In [0]:
spark.sql(f"""
    ALTER TABLE {silver_table}
    CLUSTER BY (AREA, YEAR)
""")